# 02 · Layer 1: label warnings

**Owner:** Avneesh  
**Reads:** `nodes.csv`, openFDA drug labels  
**Writes:** `data/processed/layer1_edges.csv`

Edge *i → j* when the drug-interactions section of *i*'s most recent label names *j* (directly or through a drug class), after removing "no interaction" sentences.

In [ ]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
REFERENCE = ROOT / "data" / "reference"
PROCESSED = ROOT / "data" / "processed"
OUTPUTS = ROOT / "outputs"

import os

from dotenv import load_dotenv

load_dotenv(ROOT / ".env")
API_KEY = os.getenv("OPENFDA_API_KEY", "")
print("API key found" if API_KEY else "No API key: add it to .env (see .env.example)")

In [ ]:
import sys

sys.path.insert(0, str(ROOT / "notebooks"))
import openfda_utils as fu

## Step 1 · Load the node set

In [ ]:
nodes = pd.read_csv(PROCESSED / "nodes.csv", dtype={"rxcui": str})
names = dict(zip(nodes.node_id, nodes.fda_names.str.split("|")))
print(f"{len(nodes)} drugs")

## Step 2 · One label per drug

For each drug, fetch the 10 most recent labels that list the drug as an ingredient and have a drug-interactions section, then keep the most recent **single-ingredient** one. (Labels are long, so this step downloads a few hundred MB into `data/raw/`; it only happens once.) Over-the-counter products (e.g. aspirin) and some biologics have no such section: those drugs can receive warnings but cannot give any.

The labels also give the drug's FDA classes (`pharm_class_epc` = established pharmacologic class, `pharm_class_moa` = mechanism of action), used in Step 3 for class mentions such as "NSAIDs" or "CYP3A4 inhibitors". Many generic labels carry no class data, so the classes are pooled over all 10 labels fetched for the drug.

In [ ]:
texts, classes, rows = {}, {}, []
for i, fda_names in names.items():
    q = "(" + fu.or_query("openfda.substance_name.exact", fda_names) + ")+AND+_exists_:drug_interactions"
    resp = fu.openfda(RAW, API_KEY, "label", {"search": q, "limit": 10, "sort": "effective_time:desc"}, f"label_{nodes.ingredient[i]}")
    label = fu.pick_label(resp.get("results", []))
    cls = fu.label_classes(resp.get("results", []))   # classes from all 10 labels, not just the chosen one
    classes[i] = cls
    if label:
        texts[i] = " ".join(label["drug_interactions"])
    rows.append({"node_id": i, "has_label": int(label is not None),
                 "label_set_id": label.get("set_id") if label else None,
                 "label_effective_time": label.get("effective_time") if label else None,
                 "pharm_class_epc": "|".join(cls["epc"]),
                 "pharm_class_moa": "|".join(cls["moa"])})

node_labels = pd.DataFrame(rows)
node_labels.to_csv(PROCESSED / "node_labels.csv", index=False)
print(f"{len(texts)} of {len(nodes)} drugs have a usable label")
print("No usable label:", ", ".join(nodes.ingredient[node_labels.has_label == 0]))

## Step 3 · Find the warnings

Each label's interactions text is split into sentences. Sentences that say there is *no* interaction ("no dose adjustment is necessary", "had no significant effect", …) are skipped. In the rest:

* **by name** — drug *j* is named (ingredient, a salt-free FDA name, or a synonym) → edge *i → j* with `via_name = 1`;
* **by class** — a class phrase appears ("NSAIDs", "CYP3A4 inhibitors") → edge *i → j* for every drug *j* whose own label lists that class, `via_class = 1`. A label naming its own class is describing itself, so those mentions are skipped. Limitation: FDA classes do not record strength, so "strong CYP3A4 inhibitors" links to every CYP3A4 inhibitor, weak ones included.

`evidence` keeps the first sentence that produced the edge, for checking.

In [ ]:
import re

patterns = {i: re.compile(r"\b(" + "|".join(re.escape(v) for v in fu.name_variants(nodes.ingredient[i], n)) + r")\b", re.I)
            for i, n in names.items()}
layer1 = pd.DataFrame(fu.mine_labels(texts, patterns, classes),
                      columns=["from_id", "to_id", "via_name", "via_class", "class_terms", "evidence"])
layer1.to_csv(PROCESSED / "layer1_edges.csv", index=False)
print(f"{len(layer1)} edges: {layer1.via_name.sum()} by name, {(layer1.via_name == 0).sum()} by class only")

## Checks

Reciprocity = share of edges *i → j* whose reverse *j → i* also exists. Then read the evidence for a random sample of edges: every sentence should be a real warning about the two drugs. Note any false edges in the pull request so the rules in `openfda_utils.py` can be fixed.

In [ ]:
def reciprocity(df):
    e = set(zip(df.from_id, df.to_id))
    return sum((j, i) in e for i, j in e) / len(e) if e else float("nan")

n = len(nodes)
for label, df in [("by name only", layer1[layer1.via_name == 1]), ("name + class", layer1)]:
    print(f"{label:13s} edges {len(df):5d}  density {len(df) / (n * (n - 1)):.4f}  reciprocity {reciprocity(df):.3f}")

ing = nodes.set_index("node_id").ingredient
sample = layer1.sample(min(15, len(layer1)), random_state=636)
for r in sample.itertuples():
    how = "name" if r.via_name else f"class: {r.class_terms}"
    print(f"\n{ing[r.from_id]} -> {ing[r.to_id]}  ({how})\n   {r.evidence[:220]}")
print("\nAPI requests:", fu.STATS)